In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical
import numpy as np
import torch.nn.functional as F

#LTSM is usually used for time dependent sequential data, in this case, we use it because set of hyperparameters are dependent on eachother

class HyperparameterController(nn.Module):
    def __init__(self, pos_len, pos_size, emb_dim=4, hidden_dim=32):
        super(HyperparameterController, self).__init__()

        # # of hidden layers
        self.pos_len = pos_len
        # size of each layer
        self.pos_size = pos_size
        # cap on layers it can add
        self.max_layers = max(pos_len)
        # size of this nn hidden layer
        self.hidden_dim = hidden_dim

        # mapping dictionaries
        self.act_map = {0: 'relu', 1: 'sigmoid', 2: 'softmax', 3: 'gauss'}
        self.lr_map = {0: 0.1, 1: 0.5, 2: 1.0, 3: 2.0}
        self.layers_map = {i: val for i, val in enumerate(pos_len)}
        self.size_map = {i: val for i, val in enumerate(pos_size)}

        # get length of vector for embedding, given by vocab_size
        # need vector to have dimension for each hyperparameter choice
        # order is important here, don't rearrange
        self.START_IDX = 0
        self.ACT_OFFSET = 1
        self.LR_OFFSET = self.ACT_OFFSET + len(self.act_map)
        self.LAYERS_OFFSET = self.LR_OFFSET + len(self.lr_map)
        self.SIZE_OFFSET = self.LAYERS_OFFSET + len(self.layers_map)
        vocab_size = self.SIZE_OFFSET + len(self.size_map)

        # embed variables and make long short term memory (LSTM) cell
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTMCell(emb_dim, hidden_dim)

        # output heads
        self.act_head = nn.Linear(hidden_dim, 4)
        self.lr_head = nn.Linear(hidden_dim, 4)
        self.layers_head = nn.Linear(hidden_dim, len(pos_len))
        self.size_head = nn.Linear(hidden_dim, len(pos_size)) # Shared across all sequential size steps


    def sample_hyperparameters(self, temperature=1.0):
        # initialize LSTM hidden layer and cell states
        hx = torch.zeros(1, self.hidden_dim)
        cx = torch.zeros(1, self.hidden_dim)

        # start the sequence with the dummy token
        inp = torch.tensor([self.START_IDX])

        total_log_prob = 0.0
        total_entropy  = 0.0

        #activation Function
        emb = self.embedding(inp)
        hx, cx = self.lstm(emb, (hx, cx))
        logits = self.act_head(hx) / temperature
        dist = Categorical(logits=logits)
        act_idx = dist.sample()

        total_log_prob += dist.log_prob(act_idx)
        # shannon entropy H(X) = -\sum p(x) \log p(x)
        total_entropy += dist.entropy()
        inp = act_idx + self.ACT_OFFSET # Embed this choice for the next step

        # learning rate
        emb = self.embedding(inp)
        hx, cx = self.lstm(emb, (hx, cx))
        logits = self.lr_head(hx) / temperature
        dist = Categorical(logits=logits)
        lr_idx = dist.sample()

        total_log_prob += dist.log_prob(lr_idx)
        total_entropy += dist.entropy()
        inp = lr_idx + self.LR_OFFSET

        # number of hidden layers
        emb = self.embedding(inp)
        hx, cx = self.lstm(emb, (hx, cx))
        logits = self.layers_head(hx) / temperature
        dist = Categorical(logits=logits)
        lay_idx = dist.sample()

        total_log_prob += dist.log_prob(lay_idx)
        total_entropy += dist.entropy()
        inp = lay_idx + self.LAYERS_OFFSET

        num_layers = self.layers_map[lay_idx.item()]

        # dynamically loop to pick the size for each active layer
        layer_sizes = []
        for _ in range(num_layers):
            emb = self.embedding(inp)
            hx, cx = self.lstm(emb, (hx, cx))
            logits = self.size_head(hx) / temperature
            dist = Categorical(logits=logits)
            size_idx = dist.sample()

            total_log_prob += dist.log_prob(size_idx)
            total_entropy += dist.entropy()
            inp = size_idx + self.SIZE_OFFSET # Chain the size choice into the next layer

            layer_sizes.append(self.size_map[size_idx.item()])

        params = {
            'activation': self.act_map[act_idx.item()],
            'learning_rate': self.lr_map[lr_idx.item()],
            'num_layers': num_layers,
            'layer_sizes': layer_sizes
        }

        return params, total_log_prob, total_entropy

In [ ]:
def train_child_network(params, x_train, y_train, x_test, y_test, K, N_epochs=1000, cut_per=0.25):
    mid_size_array = params['layer_sizes']

    # initialize ffnn for this set of hyperparameters
    model, x_train_tensor, y_onehot, optimizer, cost_func = setup_model_and_data(
        x_train, y_train, K,
        mid_size_array,
        params['learning_rate'],
        N_epochs,
        params['activation']
    )

    cost_history = []
    acc_history = []

    # train test nn for range of specified epochs
    for epoch in range(N_epochs):
        optimizer.zero_grad()
        NN_output = model(x_train_tensor)
        cost = cost_func(NN_output, y_onehot)
        cost.backward()
        optimizer.step()

        cost_history.append(cost.item())

        predicted_class = np.argmax(NN_output.detach().numpy(), axis=1)
        accuracy = np.mean(predicted_class == y_train)
        acc_history.append(accuracy)

    test_accuracy, test_cost = evaluate_model(model, x_test, y_test, K, cost_func)

    # get custom reward
    reward = rewardFN(acc_history, cost_history, test_accuracy, test_cost, cut_per)
    return reward

In [ ]:
class NASOptimizer:
    # includes beta for the entropy bonus, and temperature for softmax flattening
    def __init__(self, pos_len, pos_size, controller_lr=0.01, beta=0.05, temperature=1.5):
        self.controller = HyperparameterController(pos_len, pos_size)
        self.optimizer = torch.optim.Adam(self.controller.parameters(), lr=controller_lr)
        self.beta = beta
        self.temperature = temperature

    def optimize(self, x_train, y_train, x_test, y_test, K, search_episodes=50, N_epochs=1000):
        baseline_reward = 0.0
        print("Starting Neural Architecture Search...")

        for episode in range(search_episodes):
            self.optimizer.zero_grad()

            params, log_prob, entropy = self.controller.sample_hyperparameters(self.temperature)
            reward = train_child_network(params, x_train, y_train, x_test, y_test, K, N_epochs)

            if episode == 0:
                baseline_reward = reward
            else:
                #exponentially weighted moving average
                baseline_reward = 0.9 * baseline_reward + 0.1 * reward

            advantage = reward - baseline_reward

            # calculate policy gradient loss with the entropy bonus
            # helps incentivize structure to look around and not get hyperfocused
            loss = (-log_prob * advantage) - (self.beta * entropy)

            loss.backward()
            self.optimizer.step()

            print(f"Episode {episode+1:02d} | Reward: {reward:.4f} | Params: {params}")

        print("\nSearch Complete. Evaluating optimal configuration...")

        with torch.no_grad():
            # Perform a greedy, deterministic forward pass (argmax instead of sampling)
            hx = torch.zeros(1, self.controller.hidden_dim)
            cx = torch.zeros(1, self.controller.hidden_dim)
            inp = torch.tensor([self.controller.START_IDX])

            # Act
            emb = self.controller.embedding(inp)
            hx, cx = self.controller.lstm(emb, (hx, cx))
            logits = self.controller.act_head(hx)
            act_probs = F.softmax(logits, dim=-1).squeeze().numpy()
            act_idx = torch.argmax(logits)
            inp = act_idx.unsqueeze(0) + self.controller.ACT_OFFSET

            # LR
            emb = self.controller.embedding(inp)
            hx, cx = self.controller.lstm(emb, (hx, cx))
            logits = self.controller.lr_head(hx)
            lr_probs = F.softmax(logits, dim=-1).squeeze().numpy()
            lr_idx = torch.argmax(logits)
            inp = lr_idx.unsqueeze(0) + self.controller.LR_OFFSET

            # Layers
            emb = self.controller.embedding(inp)
            hx, cx = self.controller.lstm(emb, (hx, cx))
            logits = self.controller.layers_head(hx)
            layers_probs = F.softmax(logits, dim=-1).squeeze().numpy()
            lay_idx = torch.argmax(logits)
            inp = lay_idx.unsqueeze(0) + self.controller.LAYERS_OFFSET

            num_layers = self.controller.layers_map[lay_idx.item()]

            # Sizes
            size_probs_list = []
            best_layer_sizes = []

            # unroll max_layers times to populate the full plotting list
            for i in range(self.controller.max_layers):
                emb = self.controller.embedding(inp)
                hx, cx = self.controller.lstm(emb, (hx, cx))
                logits = self.controller.size_head(hx)

                size_probs_list.append(F.softmax(logits, dim=-1).squeeze().numpy())
                size_idx = torch.argmax(logits)
                inp = size_idx.unsqueeze(0) + self.controller.SIZE_OFFSET

                # Only keep the sizes that belong to the active layers
                if i < num_layers:
                    best_layer_sizes.append(self.controller.size_map[size_idx.item()])

            best_params = {
                'activation': self.controller.act_map[act_idx.item()],
                'learning_rate': self.controller.lr_map[lr_idx.item()],
                'num_layers': num_layers,
                'layer_sizes': best_layer_sizes
            }

        print(f"Optimal Hyperparameters Discovered: {best_params}")

        return best_params, act_probs, lr_probs, layers_probs, size_probs_list

In [ ]:
# after search has completed, show how related different hyperparameter choices are
def plot_embedding_inner_products(controller):
    """
    Reads the final embedding weights from the HyperparameterController,
    calculates the inner products, and plots a heatmap.
    """
    # extract the embedding weights
    with torch.no_grad():
        # Shape: [vocab_size, hidden_dim]
        weights = controller.embedding.weight.detach().numpy()

    # calculate the inner products (dot product of all vector combinations)
    # in a vector embedding, inner product shows how related two concepts are
    # Shape: [vocab_size, vocab_size]
    # calculate the L2 norm (magnitude) of each embedding vector along the hidden_dim
    norms = np.linalg.norm(weights, axis=1, keepdims=True)

    # avoid division by zero just in case an embedding collapsed to absolute zero
    norms[norms == 0] = 1e-10

    # divide each vector by its magnitude
    normalized_weights = weights / norms

    # calculate the normalized inner products (Cosine Similarity)
    # Shape: [vocab_size, vocab_size]. Values strictly between -1.0 and 1.0.
    cosine_similarities = np.dot(normalized_weights, normalized_weights.T)

    # dynamically reconstruct the label names based on offsets
    vocab_size = weights.shape[0]
    labels = [""] * vocab_size

    labels[controller.START_IDX] = "START"

    for idx, val in controller.act_map.items():
        labels[controller.ACT_OFFSET + idx] = f"Act: {val}"

    for idx, val in controller.lr_map.items():
        labels[controller.LR_OFFSET + idx] = f"LR: {val}"

    for idx, val in controller.layers_map.items():
        labels[controller.LAYERS_OFFSET + idx] = f"Layers: {val}"

    for idx, val in controller.size_map.items():
        labels[controller.SIZE_OFFSET + idx] = f"Size: {val}"

    # plot the heatmap
    fig, ax = plt.subplots(figsize=(10, 8))

    # use a diverging colormap if values center around 0
    cax = ax.imshow(cosine_similarities, cmap='coolwarm', interpolation='nearest', vmin=-1.0, vmax=1.0)
    fig.colorbar(cax, ax=ax, label='Cosine Similarity')

    # set tick marks to align with our labels
    ax.set_xticks(np.arange(vocab_size))
    ax.set_yticks(np.arange(vocab_size))

    # apply the string labels
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
    ax.set_yticklabels(labels, fontsize=10)

    ax.set_title("Hyperparameter Embedding Inner Products", fontsize=14, pad=20)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_embedding_magnitudes(controller):
    """
    Reads the final embedding weights from the HyperparameterController,
    calculates the L2 norm (magnitude) of each vector, and plots them as a bar chart.
    """
    # extract the embedding weights
    with torch.no_grad():
        # Shape: [vocab_size, hidden_dim]
        weights = controller.embedding.weight.detach().numpy()

    # calculate the L2 norm (magnitude) for each token's embedding vector
    magnitudes = np.linalg.norm(weights, axis=1)

    # dynamically reconstruct the label names based on offsets
    vocab_size = weights.shape[0]
    labels = [""] * vocab_size

    labels[controller.START_IDX] = "START"

    for idx, val in controller.act_map.items():
        labels[controller.ACT_OFFSET + idx] = f"Act: {val}"

    for idx, val in controller.lr_map.items():
        labels[controller.LR_OFFSET + idx] = f"LR: {val}"

    for idx, val in controller.layers_map.items():
        labels[controller.LAYERS_OFFSET + idx] = f"Layers: {val}"

    for idx, val in controller.size_map.items():
        labels[controller.SIZE_OFFSET + idx] = f"Size: {val}"

    # plot the magnitudes as a bar chart
    fig, ax = plt.subplots(figsize=(12, 6))

    # create bars and assign colors based on hyperparameter category
    colors = []
    for label in labels:
        if label.startswith("Act"):
            colors.append('skyblue')
        elif label.startswith("LR"):
            colors.append('lightgreen')
        elif label.startswith("Layers"):
            colors.append('salmon')
        elif label.startswith("Size"):
            colors.append('plum')
        else:
            colors.append('gray') # For the START token

    bars = ax.bar(labels, magnitudes, color=colors, edgecolor='black')

    # add exact values on top of the bars for easier reading
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 0.05, f"{yval:.2f}",
                ha='center', va='bottom', fontsize=9, rotation=90)

    ax.set_ylabel('Embedding Magnitude (L2 Norm)')
    ax.set_title('Hyperparameter Embedding Magnitudes (Feature Importance)', fontsize=14, pad=20)
    ax.set_xticks(np.arange(vocab_size))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)

    # add a horizontal grid for readability
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)

    # Optional: Adjust y-limit to make room for the text labels
    plt.ylim(0, max(magnitudes) * 1.2)

    plt.tight_layout()
    plt.show()